# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# We convert timestamp from Unix seconds to timestamp type
from pyspark.sql.functions import from_unixtime


from pyspark.sql.functions import avg, sum as spark_sum, abs as spark_abs, coalesce, lit


In [2]:
# We initialize the Spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0") \
    .getOrCreate()


In [3]:
# We define the first dataframe with raw stream
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# We define a simple schema with only useful fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# We apply schema for organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [5]:
# We make a simple aggregation for count the bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [6]:

# We use console output for have result in Docker logs
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## Exploration of edit size

We calculate the difference between the new size and the old size of page. We use absolute value for see if the modification is big, also when text was removed.

In [7]:
enriched_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp")).cast("timestamp")) \
    .withColumn(
        "edit_size",
        coalesce(col("length.new"), lit(0)) - coalesce(col("length.old"), lit(0))
    ) \
    .withColumn("absolute_edit_size", spark_abs(col("edit_size")))

## Tumbling window

This window have a duration of five minutes. The windows don't go one on the other, so one edit is count only in one window.

In [8]:
tumbling_edits_df = enriched_df \
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count"),
        avg("absolute_edit_size").alias("average_edit_size"),
        spark_sum("absolute_edit_size").alias("total_edit_size")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count"),
        col("average_edit_size"),
        col("total_edit_size")
    )

tumbling_query = tumbling_edits_df \
    .writeStream \
    .format("console") \
    .queryName("tumbling_edit_sizes") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## Overlapping window

This window have ten minutes duration and a new window start each five minutes. So the windows are overlapping and one edit can be in two results.

In [9]:
overlapping_edits_df = enriched_df \
    .groupBy(
        window(col("event_time"), "10 minutes", "5 minutes"),
        col("bot")
    ) \
    .agg(
        count("*").alias("edit_count"),
        avg("absolute_edit_size").alias("average_edit_size"),
        spark_sum("absolute_edit_size").alias("total_edit_size")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count"),
        col("average_edit_size"),
        col("total_edit_size")
    )

overlapping_query = overlapping_edits_df \
    .writeStream \
    .format("console") \
    .queryName("overlapping_edit_sizes") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## Large edits

We display also the edits who add or remove minimum 500 characters. It is useful for see quickly the biggest modifications.

In [10]:
large_edits_df = enriched_df \
    .filter(col("absolute_edit_size") >= 500) \
    .select(
        "event_time",
        "title",
        "user",
        "bot",
        "edit_size",
        "absolute_edit_size"
    )

large_edits_query = large_edits_df \
    .writeStream \
    .format("console") \
    .queryName("large_edits") \
    .outputMode("append") \
    .option("truncate", False) \
    .start()

## Producer filters

By default the producer follow `fr.wikipedia.org`. With `--wiki` we can choose another language, and with `--page` we keep only titles who contain the word.

```bash
python wikistream_producer.py --wiki en.wikipedia.org
python wikistream_producer.py --page Paris
```

We can see the results of the four streaming queries with `docker logs -f pyspark_notebook`.